# Reproducibility Audit — fastText (EACL 2017)

**Session 6, UNMSM · Initial checks**

This notebook records the initial tests; it does not yet reproduce all experiments in the paper. Run the cells in order and keep their outputs as evidence for the report.

Paper: https://aclanthology.org/E17-2068/  
Official GitHub: https://github.com/facebookresearch/fastText


## Step 1: Record the Google Colab environment

Before installing anything, we check the versions of Python and the compiler.

In [1]:
import sys, platform, subprocess
print("Python:", sys.version)
print("System:", platform.platform())
for cmd in (["git", "--version"], ["g++", "--version"]):
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(" ".join(cmd), "→", result.stdout.splitlines()[0] if result.returncode == 0 and result.stdout else result.stderr.strip())


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
System: Linux-6.6.122+-x86_64-with-glibc2.39
git --version → git version 2.43.0
g++ --version → g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Step 2: Download the original repository (after checking Step 1)

We keep the official URL. The repository was archived and is read-only; this normally does not prevent cloning it.

In [2]:
import os, subprocess
repo = "/content/fastText"
if not os.path.isdir(repo):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/facebookresearch/fastText.git", repo], check=True)
print("Repository:", repo)
result = subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, capture_output=True, text=True, check=True)
print("Exact commit:", result.stdout.strip())


Repository: /content/fastText
Exact commit: 1142dc4c4ecbc19cc16eee5cdd28472e689267e6


## Step 3: Compile fastText (C++ command-line tool)

The current Colab version uses Python 3.13. To avoid possible problems with Python bindings, we first compile the original command-line program using `make`, as described in the official README. Compilation was confirmed in Colab (evidence E03). If you restart the session, you must compile again.

In [3]:
import subprocess, os
repo = "/content/fastText"
assert os.path.isdir(repo), "First download the repository"
r = subprocess.run(["make", "-j2"], cwd=repo, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print("Last compilation lines:\n", "\n".join(r.stdout.splitlines()[-20:]))
print("Exit code:", r.returncode)
if r.returncode == 0:
    v = subprocess.run(["./fasttext"], cwd=repo, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print("Tool compiled. Help output begins:\n", "\n".join(v.stdout.splitlines()[:12]))
else:
    print("Compilation incomplete: keep this output as evidence and review the error.")


Last compilation lines:
 c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/args.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/autotune.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/matrix.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/dictionary.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/loss.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/productquantizer.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/densematrix.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/quantmatrix.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/vector.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/model.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/utils.cc
c++ -pthread -std=c++17 -march=native -O3 

## Step 4: Download and verify AG News (before training)

**CSV source:** public mirror `mhjabreel/CharCnn_Keras`, identified by URL and matching MD5 hashes in the **TorchText AG_NEWS 0.18.0** source code. The MD5 hashes in the next cell come from the TorchText technical documentation. The dataset has 120,000 training records, 7,600 test records, and 4 classes.

**Audit note:** the original fastText script used Google Drive. The mirror is documented, but byte-for-byte equivalence with the historical archive has not been verified. The original **normalization** procedure is in `classification-results.sh`, and we apply it in *Step 5*. **Do not run the full `classification-results.sh` script:** it downloads seven additional datasets.

Technical source: https://docs.pytorch.org/text/stable/_modules/torchtext/datasets/ag_news.html
Paper script: https://github.com/facebookresearch/fastText/blob/main/classification-results.sh

Keep a screenshot of the next cell as evidence **E04**. We are not training yet.

In [4]:
from pathlib import Path
from urllib.request import urlretrieve
from collections import Counter
import hashlib, csv

base = Path('/content/fastText/data/ag_news_csv')
base.mkdir(parents=True, exist_ok=True)
url_base = 'https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/'
# MD5 y records documentados en el código fuente de TorchText AG_NEWS 0.18.0
esperado = {
    'train.csv': ('b1a00f826fdfbd249f79597b59e1dc12', 120000, 30000),
    'test.csv':  ('d52ea96a97a2d943681189a97654912d',   7600,  1900),
}
for nombre, (md5_esperado, filas_esperadas, por_clase) in esperado.items():
    archivo = base / nombre
    if not archivo.exists():
        print('Downloading:', url_base + nombre)
        urlretrieve(url_base + nombre, archivo)
    md5_real = hashlib.md5(archivo.read_bytes()).hexdigest()
    assert md5_real == md5_esperado, f'MD5 incorrecto para {nombre}: {md5_real}'
    with archivo.open(encoding='utf-8', newline='') as f:
        filas = list(csv.reader(f))
    assert len(filas) == filas_esperadas, f'Cantidad inesperada: {len(filas)}'
    assert all(len(row) == 3 for row in filas), 'El CSV no contiene 3 columnas en todos los records'
    distribucion = Counter(row[0] for row in filas)
    assert distribucion == {str(k): por_clase for k in range(1, 5)}, distribucion
    print(f'{nombre}: {len(filas):,} registros | 4 clases | MD5 OK')
    print('Distribution:', dict(sorted(distribucion.items())))
    print('First title:', filas[0][1][:100])

print('STEP 4A COMPLETE: dataset downloaded and verified.')
print('Pending: apply the original fastText normalization (Step 5).')


Downloading: https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv
train.csv: 120,000 records | 4 classes | MD5 OK
Distribution: {'1': 30000, '2': 30000, '3': 30000, '4': 30000}
First title: Wall St. Bears Claw Back Into the Black (Reuters)
Downloading: https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv
test.csv: 7,600 records | 4 classes | MD5 OK
Distribution: {'1': 1900, '2': 1900, '3': 1900, '4': 1900}
First title: Fears for T N pension after talks
STEP 4A COMPLETE: dataset downloaded and verified.
Pending: apply the original fastText normalization (Step 5).


## Step 5: Normalize and shuffle using the functions in the original fastText script

In the official repository, `classification-results.sh` defines `myshuf()` and `normalize_text()`. The next cell **extracts these functions from the cloned code** and runs them on AG News, without downloading other datasets or training the model.

We check 120,000 training lines, 7,600 test lines, and the four labels `__label__1` to `__label__4`. **The original script shuffles without an explicit random seed**, so the order can change. Save a screenshot as evidence **E05**.

**Important for the audit:** the same script states that learning rates were selected through validation on part of the training data. We must not say that the paper used no validation.

In [5]:
from pathlib import Path
from collections import Counter
import subprocess

repo = Path('/content/fastText')
original = repo / 'classification-results.sh'
assert original.exists(), 'Original repository script is missing'
script = original.read_text(encoding='utf-8')
assert 'myshuf() {' in script and 'normalize_text() {' in script
assert '\nDATASET=(' in script, 'Unexpected format in original script'
# EXTRAEMOS el código exacto de las funciones originales, sin ejecutarlo entero.
funciones = script.split('\nDATASET=(', 1)[0]
orden = 'set -euo pipefail\n' + funciones + '\nnormalize_text\n'

for nombre, esperado in [('train', 120000), ('test', 7600)]:
    entrada = repo / 'data' / 'ag_news_csv' / (nombre + '.csv')
    salida = repo / 'data' / ('ag_news.' + nombre)
    assert entrada.exists(), f'Falta {entrada}: ejecuta Paso 4'
    with entrada.open('rb') as f_in, salida.open('wb') as f_out:
        subprocess.run(['bash', '-c', orden], stdin=f_in, stdout=f_out,
                       stderr=subprocess.PIPE, check=True)
    clases = Counter()
    with salida.open('rb') as f:
        for linea in f:
            etiqueta = linea.split(maxsplit=1)[0].decode('ascii')
            assert etiqueta in {'__label__1','__label__2','__label__3','__label__4'}, etiqueta
            clases[etiqueta] += 1
    assert sum(clases.values()) == esperado, f'Cantidad inesperada para {nombre}'
    assert len(clases) == 4
    print(f'{salida.name}: {sum(clases.values()):,} líneas | clases: {dict(sorted(clases.items()))}')
    print('Example:', salida.open(encoding='utf-8').readline()[:180].strip())

print('STEP 5 COMPLETE: data normalized using the original script functions.')
print('Nota: los autores utilizan shuffle sin semilla; el orden de lines puede variar.')


ag_news.train: 120,000 lines | classes: {'__label__1': 30000, '__label__2': 30000, '__label__3': 30000, '__label__4': 30000}
Example: __label__3 , netflix sees break even in 2005 , stock down 35 pct , online dvd rental company netflix inc . on thursday cut its subscription price and said it was changing its strat
ag_news.test: 7,600 lines | classes: {'__label__1': 1900, '__label__2': 1900, '__label__3': 1900, '__label__4': 1900}
Example: __label__4 , verisign bundles authentication tools , unified support for passwords , smart cards and tokens means better network security , the company says .
STEP 5 COMPLETE: data normalized using the original script functions.
Nota: los autores utilizan shuffle sin semilla; el orden de lines puede variar.


## Step 6: Train and evaluate fastText using the original settings (AG News)

**Status:** pending execution. The authors use the following settings in `classification-results.sh`: `-dim 10 -lr 0.25 -wordNgrams 2 -minCount 1 -bucket 10000000 -epoch 5 -thread 4`. We do not introduce another train/validation/test split.

Training uses `data/ag_news.train`, and final evaluation is performed once on `data/ag_news.test`. For the report, capture **E06** with the exit code, metrics, and time, and save the complete notebook with its outputs.

**Separate references:** the EACL 2017 paper, Table 1, reports **92.5%** for fastText + bigrams on AG News; the documentation for the pretrained model reports **92.4%**. Our reproduction is compared mainly with the paper table. Shuffling without a seed and asynchronous execution can cause differences between runs.

In [6]:
from pathlib import Path
import subprocess, time

repo = Path('/content/fastText')
exe = repo / 'fasttext'
train = repo / 'data/ag_news.train'
test = repo / 'data/ag_news.test'
model_dir = repo / 'result'
model_dir.mkdir(exist_ok=True)
model = model_dir / 'ag_news'

for p in (exe, train, test):
    assert p.exists(), f'No se encontró {p}; completa los pasos anteriores.'

cmd = [str(exe), 'supervised', '-input', str(train), '-output', str(model),
       '-dim', '10', '-lr', '0.25', '-wordNgrams', '2', '-minCount', '1',
       '-bucket', '10000000', '-epoch', '5', '-thread', '4']
print('Training with settings from the original script:')
print(' '.join(cmd))
start = time.perf_counter()
r = subprocess.run(cmd, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('Código de salida del entrenamiento:', r.returncode)
print('Últimas lines del entrenamiento:\n' + '\n'.join(r.stdout.splitlines()[-10:]))
assert r.returncode == 0, 'Falló el entrenamiento; conserva la salida y envíala para revisarla.'
assert (model_dir / 'ag_news.bin').exists(), 'The model file was not generated.'
print(f'Tiempo de entrenamiento: {time.perf_counter()-start:.1f} s')

prueba = subprocess.run([str(exe), 'test', str(model_dir / 'ag_news.bin'), str(test)],
                       cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('Test exit code:', prueba.returncode)
print('Results on 7,600 news articles:\n'+prueba.stdout)
assert prueba.returncode == 0, 'Test failed; save the result for review.'

print('Paper reference (Table 1, fastText + bigram, AG News): accuracy 92.5%')
print('Note: pretrained model figure reported in documentation: 92.4%')
print('STEP 6 COMPLETE: training and evaluation finished. Save E06.')


Training with settings from the original script:
/content/fastText/fasttext supervised -input /content/fastText/data/ag_news.train -output /content/fastText/result/ag_news -dim 10 -lr 0.25 -wordNgrams 2 -minCount 1 -bucket 10000000 -epoch 5 -thread 4
Código de salida del entrenamiento: 0
Últimas lines del entrenamiento:
Progress:  84.3% words/sec/thread:  945982 lr:  0.039374 avg.loss:  0.189550 ETA:   0h 0m 1s
Progress:  86.3% words/sec/thread:  954195 lr:  0.034178 avg.loss:  0.186419 ETA:   0h 0m 1s
Progress:  88.5% words/sec/thread:  963328 lr:  0.028653 avg.loss:  0.182454 ETA:   0h 0m 0s
Progress:  90.8% words/sec/thread:  972402 lr:  0.023072 avg.loss:  0.178652 ETA:   0h 0m 0s
Progress:  92.9% words/sec/thread:  980588 lr:  0.017635 avg.loss:  0.172698 ETA:   0h 0m 0s
Progress:  95.0% words/sec/thread:  988053 lr:  0.012380 avg.loss:  0.168871 ETA:   0h 0m 0s
Progress:  97.1% words/sec/thread:  995135 lr:  0.007161 avg.loss:  0.165885 ETA:   0h 0m 0s
Progress:  99.3% words/sec/